In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # disable GPU devices
os.environ["TFDS_DATA_DIR"] = os.path.expanduser("~/tensorflow_datasets")  # default location of tfds database
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers, models, metrics, losses
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

import tensorflow as tf
import tensorflow_datasets as tfds

import numpy as np

import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.manifold import TSNE

# Turn off logging for TF
import logging
logging.disable(logging.WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"
tf.get_logger().setLevel(logging.ERROR)

from dpmhm.datasets import preprocessing, feature, utils, transformer

ds_all, ds_info = tfds.load(
    'CWRU',
    with_info=True,
)

ds0 = ds_all['train']
ds0.element_spec

Load registered datasets (if not registered, run creation_metadataset.ipynb)

In [ ]:
outdir = Path('/volatile/home/bm279471/tmp/meta_dataset')
# outdir = Path('/volatile/home/bm279471/tmp/meta_dataset_full_bandwidth')
# outdir = Path('/volatile/home/bm279471/tmp/few_shot_cwru')
os.makedirs(outdir, exist_ok=True)

In [ ]:
ds_train = tf.data.Dataset.load(str(outdir/'ds_train'))
ds_val = tf.data.Dataset.load(str(outdir/'ds_val'))
ds_train_ft = tf.data.Dataset.load(str(outdir/'ds_train_ft'))
ds_val_ft = tf.data.Dataset.load(str(outdir/'ds_val_ft'))
ds_test_ft = tf.data.Dataset.load(str(outdir/'ds_test_ft'))

with open(outdir/'lb1.json', 'r') as fp:
    lb1 = list(json.load(fp))
with open(outdir/'lb2.json', 'r') as fp:
    lb2 = list(json.load(fp))
with open(outdir/'lb3.json', 'r') as fp:
    lb3 = list(json.load(fp))

In [ ]:
batch_size = 32
ds_test_size = utils.get_dataset_size(ds_train_ft)+utils.get_dataset_size(ds_val_ft)+utils.get_dataset_size(ds_test_ft)
ds_train_size = utils.get_dataset_size(ds_train)+utils.get_dataset_size(ds_val)
n_embedding  = 128 
kernel_size = (3,3)
projection_dim = 128
nb_classes=len(lb1)

In [ ]:
ds_train_clr = ds_train.map(lambda x,l:(x,x)).shuffle(ds_train_size, reshuffle_each_iteration=False).cache().batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
ds_val_clr = ds_val.cache().batch(batch_size,drop_remainder=True)
ds_train_ft= ds_train_ft.shuffle(ds_test_size, reshuffle_each_iteration=False).cache().batch(batch_size,drop_remainder=True).prefetch(tf.data.AUTOTUNE)
ds_val_ft = ds_val_ft.cache().batch(batch_size,drop_remainder=True)
ds_test_ft=ds_test_ft.cache().batch(batch_size, drop_remainder=True)

eles = list(ds_train.take(1).as_numpy_iterator())
input_shape = eles[0][0].shape

In [ ]:
tf.config.run_functions_eagerly(True)

@tf.keras.utils.register_keras_serializable()
class AddPositionEmbs(layers.Layer):
    """Adds (optionally learned) positional embeddings to the inputs."""

    def build(self, input_shape):
        assert (
            len(input_shape) == 3
        ), f"Number of dimensions should be 3, got {len(input_shape)}"
        self.pe = tf.Variable(
            name="pos_embedding",
            initial_value=tf.random_normal_initializer(stddev=0.06)(
                shape=(1, input_shape[1], input_shape[2])
            ),
            dtype="float32",
            trainable=True,
        )

    def call(self, inputs):
        return inputs + tf.cast(self.pe, dtype=inputs.dtype)

    def get_config(self):
        config = super().get_config()
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

@tf.keras.utils.register_keras_serializable()
class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, *args, num_heads, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_heads = num_heads

    def build(self, input_shape):
        embedding_dim = input_shape[-1]
        num_heads = self.num_heads
        if embedding_dim % num_heads != 0:
            raise ValueError(
                f"embedding dimension = {embedding_dim} should be divisible by number of heads = {num_heads}"
            )
        self.embedding_dim = embedding_dim
        self.projection_dim = embedding_dim // num_heads
        self.query_dense = layers.Dense(embedding_dim, name="query")
        self.key_dense = layers.Dense(embedding_dim, name="key")
        self.value_dense = layers.Dense(embedding_dim, name="value")
        self.combine_heads = layers.Dense(embedding_dim, name="out")

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], score.dtype)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)
        query = self.separate_heads(query, batch_size)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)

        attention, weights = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embedding_dim))
        output = self.combine_heads(concat_attention)
        return output, weights

    def get_config(self):
        config = super().get_config()
        config.update({"num_heads": self.num_heads})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)


@tf.keras.utils.register_keras_serializable()
class TransformerBlock(layers.Layer):
    """Implements a Transformer block."""

    def __init__(self, *args, num_heads, mlp_dim, dropout, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_heads = num_heads
        self.mlp_dim = mlp_dim
        self.dropout = dropout

    def build(self, input_shape):
        self.att = MultiHeadSelfAttention(
            num_heads=self.num_heads,
            name="MultiHeadDotProductAttention_1",
        )
        self.mlpblock = keras.Sequential(
            [
                layers.Dense(
                    self.mlp_dim,
                    activation="linear",
                    name=f"{self.name}_Dense_0",
                ),
                layers.Lambda(
                    lambda x: keras.activations.gelu(x, approximate=False)
                )
                if hasattr(keras.activations, "gelu")
                else layers.Lambda(
                    lambda x: tf.nn.gelu(x, approximate=False)
                ),
                layers.Dropout(self.dropout),
                layers.Dense(input_shape[-1], name=f"{self.name}_Dense_1"),
                layers.Dropout(self.dropout),
            ],
            name="MlpBlock_3",
        )
        self.layernorm1 = layers.LayerNormalization(
            epsilon=1e-6, name="LayerNorm_0"
        )
        self.layernorm2 = layers.LayerNormalization(
            epsilon=1e-6, name="LayerNorm_2"
        )
        self.dropout_layer = layers.Dropout(self.dropout)

    def call(self, inputs, training):
        x = self.layernorm1(inputs)
        x, weights = self.att(x)
        x = self.dropout_layer(x, training=training)
        x = x + inputs
        y = self.layernorm2(x)
        y = self.mlpblock(y)
        return x + y, weights

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "num_heads": self.num_heads,
                "mlp_dim": self.mlp_dim,
                "dropout": self.dropout,
            }
        )
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

@tf.keras.utils.register_keras_serializable()
class MultiHeadAttention(layers.Layer):
    def __init__(self, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads

    def build(self, input_shape):
        self.embedding_dim = input_shape[-1]
        assert self.embedding_dim % self.num_heads == 0

        self.projection_dim = self.embedding_dim // self.num_heads
        self.query_dense = layers.Dense(self.embedding_dim, name="query")
        self.key_dense = layers.Dense(self.embedding_dim, name="key")
        self.value_dense = layers.Dense(self.embedding_dim, name="value")
        self.combine_heads = layers.Dense(self.embedding_dim, name="out")

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], score.dtype)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs, encoder_outputs):
        batch_size = tf.shape(inputs)[0]
        
        query = self.query_dense(inputs)
        query = self.separate_heads(query, batch_size)
        
        key = self.key_dense(encoder_outputs)
        value = self.value_dense(encoder_outputs)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)

        attention, weights = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embedding_dim))
        output = self.combine_heads(concat_attention)
        return output, weights

    def get_config(self):
        config = super().get_config()
        config.update({"num_heads": self.num_heads})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)
    
@tf.keras.utils.register_keras_serializable()
class TransformerDecoderBlock(layers.Layer):
    def __init__(self, num_heads, mlp_dim, dropout, embedding_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.mlp_dim = mlp_dim
        self.dropout = dropout
        self.attention1 = MultiHeadSelfAttention(num_heads=self.num_heads, name="SelfAttention")
        self.attention2 = MultiHeadAttention(num_heads=self.num_heads, name="CrossAttention")
        
        self.mlpblock = keras.Sequential(
            [
                layers.Dense(self.mlp_dim, activation="linear", name="Dense_0"),
                layers.Lambda(
                    lambda x: keras.activations.gelu(x, approximate=False)
                ) if hasattr(keras.activations, "gelu") else layers.Lambda(
                    lambda x: tf.nn.gelu(x, approximate=False)
                ),
                layers.Dropout(self.dropout),
                layers.Dense(embedding_dim, name="Dense_1"),
                layers.Dropout(self.dropout),
            ],
            name="MlpBlock",
        )
        
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6, name="LayerNorm_0")
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6, name="LayerNorm_1")
        self.layernorm3 = layers.LayerNormalization(epsilon=1e-6, name="LayerNorm_2")
        
        self.dropout_layer = layers.Dropout(self.dropout)

    def call(self, inputs, encoder_outputs, training=False):
        x = self.layernorm1(inputs)
        attention_out1, _ = self.attention1(x)
        x = x + self.dropout_layer(attention_out1, training=training)
        
        y = self.layernorm2(x)
        attention_out2, weights = self.attention2(y, encoder_outputs)
        x = x + self.dropout_layer(attention_out2, training=training)
        
        z = self.layernorm3(x)
        mlp_output = self.mlpblock(z)
        return x + self.dropout_layer(mlp_output, training=training), weights

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_heads": self.num_heads,
            "mlp_dim": self.mlp_dim,
            "dropout": self.dropout,
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)
    
@tf.keras.utils.register_keras_serializable()
class ExtractPatches(layers.Layer):
    def __init__(self, mask_ratio=0.6, batch_size=32, patch_size=16,image_size=(64,64), **kwargs):
        super(ExtractPatches, self).__init__(**kwargs)
        self.mask_ratio = mask_ratio
        self.batch_size=batch_size
        self.image_size=image_size
        self.patch_size=patch_size

    def call(self, y):
        sub_tensors = tf.image.extract_patches(
                images=y,
                sizes=[1, self.patch_size, self.patch_size, 1],
                strides=[1, self.patch_size, self.patch_size, 1],
                rates=[1, 1, 1, 1],
                padding='VALID'
            )

        patches_per_image = (self.image_size[0] // self.patch_size) * (self.image_size[1] // self.patch_size)
        sub_tensors = tf.reshape(sub_tensors, (self.batch_size, patches_per_image, self.patch_size, self.patch_size, 3))

        nb_non_masked_patches = int((1-self.mask_ratio)*patches_per_image)
        all_indices = tf.range(patches_per_image)
        shuffled_indices = tf.random.shuffle(all_indices)
        indices = shuffled_indices[:nb_non_masked_patches]
        selected_sub_tensors = tf.gather(sub_tensors, indices, axis=1)
        selected_sub_tensors=tf.reshape(selected_sub_tensors, (-1, self.patch_size, self.patch_size, 3))

        #rest of the sub-tensors
        mask = tf.reduce_any(tf.equal(tf.expand_dims(all_indices, 1), indices), axis=1)
        remaining_indices = tf.boolean_mask(all_indices, ~mask)

        remaining_sub_tensors = tf.gather(sub_tensors, remaining_indices, axis=1)

        return selected_sub_tensors, remaining_sub_tensors, remaining_indices # shape (batch_size*nb_(non)_masked_patches, patch_size, patch_size, 3)

@tf.keras.utils.register_keras_serializable()
class MAE_ASTModel(models.Model):
    def __init__(self, image_size=(64,64), patch_size=16, enc_layers=2, dec_layers=1, num_heads=3, dropout=0.1, mlp_dim=128, mask_ratio=0.35, batch_size=32, Lambda=10, **kwargs):
        super(MAE_ASTModel, self).__init__(**kwargs)
        self.nb_patches = (image_size[0]//patch_size)*(image_size[1]//patch_size)
        self.nb_non_masked_patches=int(self.nb_patches*(1-mask_ratio))
        self.nb_masked_patches=self.nb_patches-self.nb_non_masked_patches
        self.image_size = image_size
        self.patch_size = patch_size
        self.enc_layers = enc_layers
        self.dec_layers = dec_layers
        self.embedding_dim = 3 * patch_size**2
        self.num_heads = num_heads
        self.mlp_dim = mlp_dim
        self.batch_size=batch_size
        self.Lambda=Lambda
        self.mask_ratio=mask_ratio

        self.extract_patches = ExtractPatches(mask_ratio=mask_ratio, batch_size=batch_size, patch_size=patch_size, image_size=image_size)
        self.learnable_mask = self.add_weight(
            shape=(self.batch_size*self.nb_masked_patches, patch_size, patch_size, 3),
            initializer="glorot_uniform",  
            trainable=True,
            name="learnable_mask",
        )

        self.embedding=layers.Conv2D(
            filters=self.embedding_dim,
            kernel_size=self.patch_size,
            strides=self.patch_size,
            padding="valid",
            name="embedding",
        )
        self.reshape_non_masked =layers.Reshape((batch_size, self.nb_non_masked_patches, self.embedding_dim))
        self.reshape_masked =layers.Reshape((batch_size, self.nb_masked_patches, self.embedding_dim))
        self.position_embedding_encoder=AddPositionEmbs(name="Encoder_pos_embed_input")

        self.encoder_transformer=[
            TransformerBlock(
                num_heads=self.num_heads,
                mlp_dim=self.mlp_dim,
                dropout=dropout,
                name=f"Transformer_encoderblock_{n}",
            ) for n in range(self.enc_layers)
        ]
        self.transformer_norm = layers.LayerNormalization(
            epsilon=1e-6, name="Transformer_encoder_norm"
        )

        self.position_embedding_decoder=AddPositionEmbs(name="Decoder_pos_embed_input")
        self.decoder_transformer=[
            TransformerDecoderBlock(
                num_heads=self.num_heads,
                mlp_dim=self.mlp_dim,
                dropout=dropout,
                embedding_dim=self.embedding_dim,
                name=f"Transformer_decoderblock_{n}",
            ) for n in range(self.dec_layers)
        ]

        self.embedding_inverse = layers.Conv2DTranspose(
            filters=3,
            kernel_size=self.patch_size,
            strides=self.patch_size,
            padding="valid",
            name="embedding_inverse",
        )
        self.reshape_inverse = layers.Reshape((self.image_size[0] // self.patch_size, self.image_size[1] // self.patch_size, self.embedding_dim))

    def convert_list_to_tensor(self, L, dtype=tf.float32):
        max_len = max(len(inner_list) for inner_list in L)
        padded_lists = []
        for inner_list in L:
            padded_list = [tf.convert_to_tensor(item, dtype=dtype) for item in inner_list]
            padded_list += [tf.zeros_like(inner_list[0], dtype=tf.float32)] * (max_len - len(inner_list))
            padded_lists.append(tf.stack(padded_list))
        return tf.stack(padded_lists)
    
    def classification(self, r_i):
        """For each masked patch, selects the correct patch among the candidates patches"""
        classifications = []
        for i in range(len(r_i)):
            batch=r_i[i]
            patch_classifications = []
            for j in range(len(batch)):
                patch = batch[j]
                candidate_similarities = []
                for candidate in batch:
                    similarity = keras.losses.cosine_similarity(patch, candidate, axis=-1)
                    similarity = tf.reshape(similarity, (self.patch_size**2))
                    candidate_similarities.append(tf.reduce_mean(similarity))
                patch_classification = tf.stack(candidate_similarities)
                patch_classifications.append(patch_classification)
            classifications.append(patch_classifications)
        c_i = self.convert_list_to_tensor(classifications)


        c_i_true = [[i for i in range(len(batch))] for _ in range(self.batch_size)] 
        c_i_true = tf.convert_to_tensor(c_i_true, dtype=tf.int64)
        return c_i, c_i_true

    def call(self, x):
        E_non_masked, r_i_true, remaining_indices =  self.extract_patches.call(x)


        E_non_masked = self.embedding(E_non_masked) # shape (batch_size*mask_ratio*nb_non_masked_patches, 1, 1, embedding_dim)
        E_non_masked = tf.expand_dims(tf.squeeze(E_non_masked, axis=2), axis=0)
        E_non_masked = tf.squeeze(self.reshape_non_masked(E_non_masked), axis=0) # shape (batch_size, mask_ratio*nb_non_masked_patches, embedding_dim)

        E_mask = self.embedding(self.learnable_mask) # shape (batch_size*mask_ratio*nb_non_masked_patches, 1, 1, embedding_dim)
        E_mask = tf.expand_dims(tf.squeeze(E_mask, axis=2), axis=0)
        E_mask = tf.squeeze(self.reshape_masked(E_mask), axis=0) # shape (batch_size, mask_ratio*nb_non_masked_patches, embedding_dim)

        O = self.position_embedding_encoder(E_non_masked)
        for n in range(self.enc_layers):
            O, _ = self.encoder_transformer[n](O, training=True)
        O = self.transformer_norm(O)

        New_O=tf.zeros((self.batch_size, self.nb_patches, self.embedding_dim))
        i_E, i_O=0,0
        for i in range(self.nb_patches):
            if i in remaining_indices:
                indices = tf.constant([[j, i] for j in range(self.batch_size)])
                updates = E_mask[:, i_E, :]
                New_O = tf.tensor_scatter_nd_update(New_O, indices, updates)
                i_E+=1
            else:
                indices = tf.constant([[j, i] for j in range(self.batch_size)])
                updates = O[:, i_O, :]
                New_O = tf.tensor_scatter_nd_update(New_O, indices, updates)
                i_O+=1

        New_O = self.position_embedding_decoder(New_O)
        for n in range(self.dec_layers):
            New_O, _ = self.decoder_transformer[n].call(New_O, O, training=True)

        r_i = tf.gather(New_O, remaining_indices, axis=1)
        r_i = tf.reshape(r_i, (-1, self.patch_size, self.patch_size, 3))
        r_i = tf.reshape(r_i, (self.batch_size, self.nb_masked_patches, self.patch_size, self.patch_size, 3))

        c_i, c_i_true=self.classification(r_i)
        return r_i, r_i_true, c_i, c_i_true

    def compile(self, optimizer, **kwargs):
        super(MAE_ASTModel, self).compile(**kwargs)
        self.optimizer = optimizer
        self.loss_tracker = metrics.Mean(name="loss")
        self.reconstruction_loss_tracker = metrics.Mean(name="recon_loss")
        self.classification_loss_tracker = metrics.Mean(name="class_loss")

    @property
    def metrics(self):
        return [self.loss_tracker, self.reconstruction_loss_tracker, self.classification_loss_tracker]

    def train_step(self, data):
        x, _ = data
        with tf.GradientTape() as tape:
            r_i, r_i_true, c_i, c_i_true = self(x, training=False)
            reconstruction_loss = self.Lambda * losses.mean_squared_error(r_i_true, r_i)
            classification_loss = tf.reduce_mean(losses.sparse_categorical_crossentropy(c_i_true, c_i, from_logits=True))
            total_loss = tf.reduce_mean(reconstruction_loss) + tf.reduce_mean(classification_loss)

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        self.loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.classification_loss_tracker.update_state(classification_loss)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, _ = data

        r_i, r_i_true, c_i, c_i_true = self(x, training=False)
        reconstruction_loss = self.Lambda * losses.mean_squared_error(r_i_true, r_i)
        classification_loss = losses.sparse_categorical_crossentropy(c_i_true, c_i, from_logits=True)
        total_loss = tf.reduce_mean(reconstruction_loss) + tf.reduce_mean(classification_loss)

        self.loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.classification_loss_tracker.update_state(classification_loss)
        return {m.name: m.result() for m in self.metrics}
    
mae_ast_model = MAE_ASTModel(image_size=(input_shape[0], input_shape[1]), Lambda=0.1)
test_input = tf.random.normal((32, input_shape[0], input_shape[1], 3))
test_output = mae_ast_model(test_input)
print(test_output[0].shape, test_output[1].shape, test_output[2].shape, test_output[3].shape)

In [ ]:
mae_ast_model.compile(
    optimizer=keras.optimizers.Adam()
)

history = mae_ast_model.fit(
    ds_train_clr.repeat(),
    validation_data=ds_val_clr,
    epochs=1,
    steps_per_epoch=(int(0.8*ds_train_size)//batch_size),
)

In [ ]:
mae_ast_model.save_weights('mae_ast_meta_dataset_full_bandwidth.weights.h5')
mae_ast_model.load_weights('mae_ast_meta_dataset_full_bandwidth.weights.h5')

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']
reconstruction_loss = history.history['recon_loss']
val_reconstruction_loss = history.history['val_recon_loss']
classification_loss = history.history['class_loss']
val_classification_loss = history.history['val_class_loss']

# Créer une figure et des axes
fig, ax = plt.subplots(figsize=(10, 6))

# Plot loss et val_loss
epochs = np.arange(1, len(loss) + 1)
ax.plot(epochs, loss, label='Training Loss', marker='o', linestyle='-', color='r')
ax.plot(epochs, val_loss, label='Validation Loss', marker='o', linestyle='--', color='r')


ax.plot(epochs, reconstruction_loss, label='Training Reconstruction Loss', marker='o', linestyle='-', color='b')
ax.plot(epochs, val_reconstruction_loss, label='Validation Reconstruction Loss', marker='o', linestyle='--', color='b')

ax.plot(epochs, classification_loss, label='Training Classification Loss', marker='o', linestyle='-', color='g')
ax.plot(epochs, val_classification_loss, label='Validation Classification Loss', marker='o', linestyle='--', color='g')

# Configurer les labels et la légende
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.set_title('Training and Validation Losses')
ax.legend()

# Afficher le graphique
plt.tight_layout()
plt.show()

In [ ]:
@tf.keras.utils.register_keras_serializable()
class ClassificationModel(models.Model):
    def __init__(self, model, **kwargs):
        super(ClassificationModel, self).__init__(**kwargs)
        self.num_layers=model.enc_layers

        self.extract_patches=model.extract_patches
        self.embedding=model.embedding
        self.position_embedding=model.position_embedding_encoder
        self.reshape=model.reshape_non_masked
        self.transformer=model.encoder_transformer
        self.transformer_norm=model.transformer_norm
        self.meanpooling = layers.GlobalAveragePooling1D(name="MeanPooling")
        self.classification_layer = layers.Dense(nb_classes, name="Classification_head")

    def _set_non_trainable_layers(self):
        for layer in [self.embedding, self.reshape, self.position_embedding, self.transformer_norm]:
            layer.trainable = False
        for transformer in self.transformer:
            transformer.trainable = False
            transformer.att.query_dense.trainable = False
            transformer.att.key_dense.trainable = False
            transformer.att.value_dense.trainable = False
            transformer.att.combine_heads.trainable = False
            transformer.mlpblock.trainable = False
            transformer.layernorm1.trainable = False
            transformer.layernorm2.trainable = False
            transformer.dropout_layer.trainable = False

    def _set_trainable_layers(self):
        for layer in [self.embedding, self.reshape, self.position_embedding, self.transformer_norm]:
            layer.trainable = True
        for transformer in self.transformer:
            transformer.trainable = True
            transformer.att.query_dense.trainable = True
            transformer.att.key_dense.trainable = True
            transformer.att.value_dense.trainable = True
            transformer.att.combine_heads.trainable = True
            transformer.mlpblock.trainable = True
            transformer.layernorm1.trainable = True
            transformer.layernorm2.trainable = True
            transformer.dropout_layer.trainable = True

    def call(self, x):
        E_non_masked, _, _ =  self.extract_patches.call(x)

        E_non_masked = self.embedding(E_non_masked) 
        E_non_masked = tf.expand_dims(tf.squeeze(E_non_masked, axis=2), axis=0)
        E_non_masked = self.reshape(E_non_masked)
        E_non_masked = tf.squeeze(E_non_masked, axis=0)

        O = self.position_embedding(E_non_masked)
        for n in range(self.num_layers):
            O, _ = self.transformer[n](O, training=True)
        O = self.transformer_norm(O)

        O = self.meanpooling(O)
        O = self.classification_layer(O)
        return O
 
classification_model=ClassificationModel(mae_ast_model)
classification_model._set_non_trainable_layers()

test_input = tf.random.normal((32, input_shape[0], input_shape[1], 3))
test_output = classification_model(test_input)
classification_model.summary(show_trainable=True, expand_nested=True)

In [ ]:
classification_model.compile(
    optimizer=keras.optimizers.Adam(1e-2), 
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
    metrics=['accuracy'])

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,
    patience=3 
)

classification_history = classification_model.fit(
    ds_train_ft.repeat(),
    validation_data=ds_val_ft,
    epochs=80,
    steps_per_epoch=(int(0.1*ds_test_size)//batch_size),
    callbacks=[reduce_lr]
)

In [ ]:
fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('Epochs')
ax1.set_ylabel('loss', color=color)
ax1.plot(classification_history.history['val_loss'], color=color, linestyle='--')
ax1.plot(classification_history.history['loss'], color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

color = 'tab:blue'
ax2.set_ylabel('acc', color=color) 
ax2.plot(classification_history.history['val_accuracy'], color=color, linestyle='--')
ax2.plot(classification_history.history['accuracy'], color=color)
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()
plt.title('Evolution of metrics')
plt.show()

In [ ]:
evaluation = classification_model.evaluate(ds_test_ft)
print("Evaluation accuracy : {:.2f}%".format(evaluation[1]*100))